# E4 — YOLO26s + DySample trên DataSet2

Notebook này là ablation độc lập cho E4. Chỉ thay hai upsampling ×2 trong neck tương ứng `P5 -> P4` và `P4 -> P3` bằng DySample; không thêm P2, detection scale hay module attention khác.

DySample bám implementation chính thức trong [tiny-smart/dysample](https://github.com/tiny-smart/dysample), paper *Learning to Upsample by Learning to Sample* (ICCV 2023). Runtime YOLO26s được inspect bằng forward hooks và shape thực tế, không đoán layer index từ model khác.

Notebook giữ nguyên dataset, split, seed và training protocol của baseline; output riêng là `helmet_yolo26s_dysample_dataset2`. Các cell validation chỉ dùng `split='val'`.

In [ ]:
# Cài đúng version baseline trong Colab runtime mới; không nâng version trôi nổi.
%pip install -q ultralytics==8.4.155
%pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
# Thiết lập dataset giống baseline; không sửa nội dung hoặc train/val split.
from google.colab import drive
import os
import shutil
from pathlib import Path
import gdown

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/DetectedHelmets/DataSet2', exist_ok=True)
os.makedirs('/content/DetectedHelmets/Code', exist_ok=True)
os.chdir('/content/DetectedHelmets')

archive_path = Path('/content/DetectedHelmets/dataset.rar')
data_root = Path('/content/DetectedHelmets/DataSet2')
drive_dataset_path = Path('/content/drive/MyDrive/DataSet2')
local_dataset_path = data_root

if drive_dataset_path.exists():
    if local_dataset_path.is_symlink():
        local_dataset_path.unlink()
    elif local_dataset_path.exists() and local_dataset_path.resolve() != drive_dataset_path.resolve():
        shutil.rmtree(local_dataset_path)
    if not local_dataset_path.exists():
        local_dataset_path.symlink_to(drive_dataset_path, target_is_directory=True)
    print(f'Dataset source: {local_dataset_path} -> {drive_dataset_path}')
else:
    if archive_path.exists():
        archive_path.unlink()
    file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(archive_path), quiet=False, fuzzy=True)
    os.system(f'unrar x -o+ {archive_path} {data_root}/')
    print(f'Dataset extracted to: {data_root}')

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics
import yaml
from PIL import Image
from ultralytics import YOLO

ULTRALYTICS_VERSION = '8.4.155'
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
assert ultralytics.__version__ == ULTRALYTICS_VERSION, (ultralytics.__version__, ULTRALYTICS_VERSION)
assert torch.cuda.is_available(), 'GPU chưa được bật; protocol yêu cầu device=0.'

## 1. Dataset, class order và training protocol

Không đưa các metric tham chiếu vào kết quả E4. Mọi số liệu ở phần sau đều được tính lại từ checkpoint E4 và cùng validation set với baseline.

In [ ]:
DATA_YAML_PATH = Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
MODEL_NAME = 'yolo26s.pt'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_dysample_dataset2'
BASELINE_EXPERIMENT_NAME = 'helmet_yolo26s_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
BASELINE_BEST_MODEL_PATH = OUTPUT_DIR / BASELINE_EXPERIMENT_NAME / 'weights' / 'best.pt'
VALIDATION_SPLIT = 'val'

TRAIN_CONFIG = {
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'workers': 4,
    'optimizer': 'auto',
    'seed': 0,
    'deterministic': True,
    'patience': 15,
    'pretrained': True,
}

assert DATA_YAML_PATH.exists(), f'Không tìm thấy data.yaml: {DATA_YAML_PATH}'
with DATA_YAML_PATH.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)
raw_names = data_config.get('names', [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
CLASS_NAMES = list(raw_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, (CLASS_NAMES, EXPECTED_CLASS_NAMES)
assert EXPERIMENT_NAME != BASELINE_EXPERIMENT_NAME

print(f'Dataset: {DATA_YAML_PATH}')
print(f'Classes: {CLASS_NAMES}')
print(f'Output: {EXPERIMENT_DIR}')
print(f'Baseline checkpoint: {BASELINE_BEST_MODEL_PATH}')
print(f'Train protocol: {TRAIN_CONFIG}')

## 2. Inspect YOLO26s runtime và tìm upsampling neck

Cell này in model graph và dùng forward hooks để ghi input/output shape của mọi `nn.Upsample` ×2 trước Detect. `input_stride -> output_stride` xác định hướng P5→P4 hoặc P4→P3; không hard-code layer index. Nếu runtime không có đúng hai điểm chuẩn, notebook dừng với danh sách layer thực tế thay vì ép thay mù quáng.

In [ ]:
base_model = YOLO(MODEL_NAME)
assert getattr(base_model, 'task', None) == 'detect', base_model.task
print('--- YOLO26s runtime architecture ---')
print(base_model.model)
print('--- YOLO26s model.info(imgsz=640) ---')
base_model.info(verbose=True, imgsz=640)

def is_scale_two_upsample(module):
    if not isinstance(module, nn.Upsample):
        return False
    scale_factor = module.scale_factor
    if isinstance(scale_factor, (tuple, list)):
        return len(scale_factor) == 2 and all(float(value) == 2.0 for value in scale_factor)
    return float(scale_factor) == 2.0

def probe_runtime_upsamples(yolo_model, imgsz=640):
    records = []
    handles = []

    def make_hook(index, module):
        def hook(_module, inputs, output):
            input_tensor = inputs[0] if inputs and torch.is_tensor(inputs[0]) else None
            if input_tensor is None or not torch.is_tensor(output):
                return
            input_shape = tuple(input_tensor.shape)
            output_shape = tuple(output.shape)
            input_stride = imgsz // input_shape[-1]
            output_stride = imgsz // output_shape[-1]
            records.append({
                'index': index,
                'from': getattr(module, 'f', None),
                'type': module.__class__.__name__,
                'mode': getattr(module, 'mode', None),
                'scale_factor': getattr(module, 'scale_factor', None),
                'input_shape': input_shape,
                'output_shape': output_shape,
                'input_channels': input_shape[1],
                'input_stride': input_stride,
                'output_stride': output_stride,
            })
        return hook

    for index, module in enumerate(yolo_model.model.model):
        if is_scale_two_upsample(module):
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()
    return sorted(records, key=lambda item: item['index'])

detect_indices = [
    index for index, module in enumerate(base_model.model.model)
    if module.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}
]
assert detect_indices, 'Không tìm thấy Detect head trong YOLO26s runtime graph.'
detect_index = min(detect_indices)
DETECT_HEAD_COUNT = len(detect_indices)

runtime_upsample_records = probe_runtime_upsamples(base_model, imgsz=640)
print('--- All runtime nn.Upsample(scale_factor=2) records ---')
for record in runtime_upsample_records:
    print(record)

neck_upsample_records = [
    record for record in runtime_upsample_records
    if record['index'] < detect_index
    and record['input_stride'] == record['output_stride'] * 2
    and record['output_stride'] in (8, 16)
]
by_output_stride = {}
for record in neck_upsample_records:
    output_stride = record['output_stride']
    if output_stride in by_output_stride:
        raise RuntimeError(
            f'Runtime có nhiều neck upsample ×2 cùng output stride {output_stride}: '
            f'{by_output_stride[output_stride]} và {record}. Không thay mù quáng.'
        )
    by_output_stride[output_stride] = record

if len(neck_upsample_records) != 2 or set(by_output_stride) != {8, 16}:
    raise RuntimeError(
        'Kiến trúc YOLO26s runtime không khớp hai neck upsample P5->P4/P4->P3. '
        f'Các record phù hợp: {neck_upsample_records}. Không ép DYSAMPLE_EXPECTED_COUNT=2.'
    )

DYSAMPLE_EXPECTED_COUNT = 2
upsample_specs = [
    {
        'level': 'P4 -> P3' if output_stride == 8 else 'P5 -> P4',
        'index': by_output_stride[output_stride]['index'],
        'from': by_output_stride[output_stride]['from'],
        'input_channels': by_output_stride[output_stride]['input_channels'],
        'input_stride': by_output_stride[output_stride]['input_stride'],
        'output_stride': output_stride,
        'input_shape': by_output_stride[output_stride]['input_shape'],
        'output_shape': by_output_stride[output_stride]['output_shape'],
    }
    for output_stride in (16, 8)
]
assert len(upsample_specs) == DYSAMPLE_EXPECTED_COUNT
print(f'Detect head runtime index: {detect_index}; count: {DETECT_HEAD_COUNT}')
print('--- Runtime-selected DySample insertion points ---')
for spec in upsample_specs:
    print(spec)

## 3. Official DySample implementation

Công thức dưới đây giữ nguyên ý tưởng và các phép biến đổi của source chính thức: learned offsets, optional scope, `init_pos`, `pixel_shuffle`, `grid_sample`, `align_corners=False`, không dynamic convolution và không custom CUDA. `groups` được chọn từ channel thực tế của từng upsample; ưu tiên `groups=4`, nếu không hợp lệ thì chọn divisor gần nhất và in rõ ra.

In [ ]:
def normal_init(module, mean=0, std=1, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.normal_(module.weight, mean, std)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)

def constant_init(module, val, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.constant_(module.weight, val)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)

class DySample(nn.Module):
    def __init__(self, in_channels, scale=2, style='lp', groups=4, dyscope=False):
        super().__init__()
        self.scale = scale
        self.style = style
        self.groups = groups
        assert style in ['lp', 'pl']
        if style == 'pl':
            assert in_channels >= scale ** 2 and in_channels % scale ** 2 == 0
        assert in_channels >= groups and in_channels % groups == 0
        if style == 'pl':
            in_channels = in_channels // scale ** 2
            out_channels = 2 * groups
        else:
            out_channels = 2 * groups * scale ** 2

        self.offset = nn.Conv2d(in_channels, out_channels, 1)
        normal_init(self.offset, std=0.001)
        if dyscope:
            self.scope = nn.Conv2d(in_channels, out_channels, 1, bias=False)
            constant_init(self.scope, val=0.)

        self.register_buffer('init_pos', self._init_pos())

    def _init_pos(self):
        h = torch.arange((-self.scale + 1) / 2, (self.scale - 1) / 2 + 1) / self.scale
        return torch.stack(torch.meshgrid([h, h])).transpose(1, 2).repeat(1, self.groups, 1).reshape(1, -1, 1, 1)

    def sample(self, x, offset):
        B, _, H, W = offset.shape
        offset = offset.reshape(B, 2, -1, H, W)
        coords_h = torch.arange(H) + 0.5
        coords_w = torch.arange(W) + 0.5
        coords = torch.stack(torch.meshgrid([coords_w, coords_h])
                             ).transpose(1, 2).unsqueeze(1).unsqueeze(0).type(x.dtype).to(x.device)
        normalizer = torch.tensor([W, H], dtype=x.dtype, device=x.device).reshape(1, 2, 1, 1, 1)
        coords = 2 * (coords + offset) / normalizer - 1
        # coords can be non-contiguous after transpose/coordinate construction on newer PyTorch.
        # contiguous + reshape preserves the official sampling computation and its values.
        coords_for_shuffle = coords.contiguous().reshape(B, -1, H, W)
        coords = torch.nn.functional.pixel_shuffle(coords_for_shuffle, self.scale).reshape(
            B, 2, -1, self.scale * H, self.scale * W).permute(0, 2, 3, 4, 1).contiguous().flatten(0, 1)
        return torch.nn.functional.grid_sample(x.reshape(B * self.groups, -1, H, W), coords, mode='bilinear',
                                               align_corners=False, padding_mode='border').reshape(B, -1, self.scale * H, self.scale * W)

    def forward_lp(self, x):
        if hasattr(self, 'scope'):
            offset = self.offset(x) * self.scope(x).sigmoid() * 0.5 + self.init_pos
        else:
            offset = self.offset(x) * 0.25 + self.init_pos
        return self.sample(x, offset)

    def forward_pl(self, x):
        x_ = torch.nn.functional.pixel_shuffle(x, self.scale)
        if hasattr(self, 'scope'):
            offset = torch.nn.functional.pixel_unshuffle(self.offset(x_) * self.scope(x_).sigmoid(), self.scale) * 0.5 + self.init_pos
        else:
            offset = torch.nn.functional.pixel_unshuffle(self.offset(x_), self.scale) * 0.25 + self.init_pos
        return self.sample(x, offset)

    def forward(self, x):
        if self.style == 'pl':
            return self.forward_pl(x)
        return self.forward_lp(x)

def choose_groups(in_channels, preferred=4):
    divisors = [group for group in range(1, in_channels + 1) if in_channels % group == 0]
    return min(divisors, key=lambda group: (abs(group - preferred), -group))

for spec in upsample_specs:
    spec['groups'] = choose_groups(spec['input_channels'], preferred=4)
    spec['style'] = 'lp'
    spec['dyscope'] = True
    print(f"{spec['level']}: C={spec['input_channels']}, groups={spec['groups']} (preferred=4)")

In [ ]:
# Focused unit checks: official DySample preserves channels, doubles spatial size and receives gradients.
assert 'view' not in DySample.sample.__code__.co_names, 'DySample class cũ còn trong runtime; hãy chạy lại cell định nghĩa class.'
unit_module = DySample(64, scale=2, style='lp', groups=4, dyscope=True)
unit_input = torch.randn(2, 64, 20, 20, requires_grad=True)
unit_output = unit_module(unit_input)
assert unit_output.shape == (2, 64, 40, 40), unit_output.shape
unit_output.mean().backward()
assert any(parameter.grad is not None for parameter in unit_module.parameters())
assert unit_module.offset.out_channels == 2 * 4 * 2 ** 2
print('DySample unit check: [B,C,H,W] -> [B,C,2H,2W], official grid_sample path and gradient OK.')

## 4. Thay đúng hai upsample trong graph và transfer pretrained weights

Các layer YOLO26s tương thích được copy từ `yolo26s.pt`. Chỉ tham số mới của hai DySample được khởi tạo theo official implementation.

In [ ]:
def copy_graph_metadata(replacement, old_module, fallback_index, fallback_from):
    replacement.i = getattr(old_module, 'i', fallback_index)
    replacement.f = getattr(old_module, 'f', fallback_from)
    replacement.type = replacement.__class__.__name__
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    return replacement

def count_dysample(module):
    return sum(1 for submodule in module.modules() if submodule.__class__.__name__ == 'DySample')

def replace_runtime_upsamples(yolo_model, specs):
    for spec in specs:
        old_module = yolo_model.model.model[spec['index']]
        assert is_scale_two_upsample(old_module), (spec, old_module.__class__.__name__)
        replacement = DySample(
            spec['input_channels'],
            scale=2,
            style=spec['style'],
            groups=spec['groups'],
            dyscope=spec['dyscope'],
        )
        yolo_model.model.model[spec['index']] = copy_graph_metadata(
            replacement, old_module, spec['index'], spec['from']
        )
    return yolo_model

def build_custom_dysample_model(model_name=MODEL_NAME):
    return replace_runtime_upsamples(YOLO(model_name), upsample_specs)

custom_model = build_custom_dysample_model()
source_state = base_model.model.state_dict()
target_state = custom_model.model.state_dict()
compatible_state = {
    key: value
    for key, value in source_state.items()
    if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)
}
load_report = custom_model.model.load_state_dict(compatible_state, strict=False)

source_parameters = dict(base_model.model.named_parameters())
target_parameters = dict(custom_model.model.named_parameters())
compatible_parameter_keys = {
    key for key in compatible_state if key in source_parameters and key in target_parameters
}
transferred_parameter_count = sum(source_parameters[key].numel() for key in compatible_parameter_keys)
new_parameter_count = sum(value.numel() for key, value in target_parameters.items() if key not in compatible_parameter_keys)
baseline_parameter_count = sum(parameter.numel() for parameter in base_model.model.parameters())
custom_parameter_count = sum(parameter.numel() for parameter in custom_model.model.parameters())

model = custom_model
print(f'Baseline parameters: {baseline_parameter_count:,}')
print(f'Custom parameters: {custom_parameter_count:,}')
print(f'Additional parameters: {custom_parameter_count - baseline_parameter_count:,}')
print(f'Pretrained parameter values transferred: {transferred_parameter_count:,}')
print(f'New DySample parameter values: {new_parameter_count:,}')
print(f'Compatible state tensors transferred: {len(compatible_state):,}')
print(f'New/unmatched state tensors: {len(load_report.missing_keys):,}')
print(f'Unexpected state keys: {load_report.unexpected_keys}')
print(f'DySample modules in graph: {count_dysample(model.model)}')
assert count_dysample(model.model) == DYSAMPLE_EXPECTED_COUNT == 2

## 5. Preflight: shape, full forward và backward gradient

Phải pass toàn bộ preflight trước khi tạo trainer hoặc bắt đầu train.

In [ ]:
for spec in upsample_specs:
    module = model.model.model[spec['index']]
    height, width = spec['input_shape'][-2:]
    module_input = torch.randn(2, spec['input_channels'], height, width)
    module_output = module(module_input)
    assert module_output.shape == (2, spec['input_channels'], height * 2, width * 2), (spec, module_output.shape)

def sum_tensors(value):
    if torch.is_tensor(value):
        return value.float().sum()
    if isinstance(value, (list, tuple)):
        terms = [sum_tensors(item) for item in value]
        return sum(terms) if terms else torch.zeros(())
    if isinstance(value, dict):
        terms = [sum_tensors(item) for item in value.values()]
        return sum(terms) if terms else torch.zeros(())
    return torch.zeros(())

device = next(model.model.parameters()).device
model.model.eval()
with torch.no_grad():
    dummy_output = model.model(torch.randn(2, 3, 640, 640, device=device))
print('Dummy forward torch.randn(2, 3, 640, 640): OK')

model.model.train()
model.model.zero_grad(set_to_none=True)
graph_input = torch.randn(2, 3, 640, 640, device=device, requires_grad=True)
graph_output = model.model(graph_input)
graph_loss = sum_tensors(graph_output)
assert graph_loss.requires_grad, 'Dummy output không nằm trong computational graph.'
graph_loss.backward()
dysample_gradients = [
    parameter.grad
    for submodule in model.model.modules()
    if submodule.__class__.__name__ == 'DySample'
    for parameter in submodule.parameters()
    if parameter.grad is not None
]
assert dysample_gradients and all(torch.isfinite(gradient).all() for gradient in dysample_gradients)
print(f'Computational graph check: OK ({len(dysample_gradients)} DySample parameter gradients)')
model.model.eval()

## 6. Custom DetectionTrainer và training E4

Trainer trả về đúng PyTorch graph đã thay DySample, không rebuild từ YAML. Cell kiểm tra module count ngay trước và ngay trong trainer trước khi gọi `train()`.

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer

custom_pytorch_model = model.model
dysample_before_train = count_dysample(custom_pytorch_model)
print('DySample BEFORE TRAIN:', dysample_before_train)
assert dysample_before_train == DYSAMPLE_EXPECTED_COUNT == 2

class CustomDySampleTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        return custom_pytorch_model

    def setup_model(self):
        self.model = custom_pytorch_model

        assert count_dysample(self.model) == 2, \
            "DySample bị mất trước training"

        print(
            "DySample INSIDE setup_model:",
            count_dysample(self.model)
        )

        return None

TRAIN_OVERRIDES = {
    'model': MODEL_NAME,
    'data': str(DATA_YAML_PATH),
    **TRAIN_CONFIG,
    'project': str(OUTPUT_DIR),
    'name': EXPERIMENT_NAME,
    'save': True,
    'save_period': 10,
    'plots': True,
    'exist_ok': False,
}

trainer = CustomDySampleTrainer(overrides=TRAIN_OVERRIDES, _callbacks=model.callbacks)
trainer.model = custom_pytorch_model
trainer_dysample_count = count_dysample(trainer.model)
print('DySample INSIDE TRAINER:', trainer_dysample_count)
assert trainer_dysample_count == DYSAMPLE_EXPECTED_COUNT == 2
print(trainer.model)

train_results = trainer.train()

EXPERIMENT_VERSION_FILE = EXPERIMENT_DIR / 'ultralytics_version.txt'
EXPERIMENT_VERSION_FILE.write_text(
    f'ultralytics=={ultralytics.__version__}\n'
    f'model={MODEL_NAME}\n'
    f'dysample_expected_count={DYSAMPLE_EXPECTED_COUNT}\n'
    f'upsample_specs={upsample_specs}\n',
    encoding='utf-8',
)
print(f'Experiment version recorded at: {EXPERIMENT_VERSION_FILE}')

## 7. Load best checkpoint mà không làm mất DySample

Ưu tiên load trực tiếp nếu Ultralytics deserialize được custom class. Nếu không, rebuild YOLO26s + DySample theo cùng runtime specs rồi load các tensor tương thích từ checkpoint.

In [ ]:
def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def extract_checkpoint_state(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('model', 'ema', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module):
                return value.state_dict()
            if isinstance(value, dict):
                tensor_state = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensor_state:
                    return tensor_state
        tensor_state = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensor_state:
            return tensor_state
    raise TypeError(f'Không tìm thấy state dict trong checkpoint type={type(payload)}')

def load_custom_dysample_checkpoint(path):
    direct_error = None
    try:
        direct_model = YOLO(str(path))
        if count_dysample(direct_model.model) > 0:
            return direct_model, 'direct'
        direct_error = RuntimeError('Direct load returned a graph without DySample.')
    except Exception as exc:
        direct_error = exc
    print(f'Direct checkpoint load unavailable: {direct_error}')

    rebuilt = build_custom_dysample_model()
    payload = torch_load_checkpoint(str(path))
    checkpoint_state = extract_checkpoint_state(payload)
    if checkpoint_state and all(name.startswith('module.') for name in checkpoint_state):
        checkpoint_state = {name[len('module.'):]: value for name, value in checkpoint_state.items()}
    target_state = rebuilt.model.state_dict()
    compatible = {
        name: value
        for name, value in checkpoint_state.items()
        if name in target_state and tuple(value.shape) == tuple(target_state[name].shape)
    }
    if not compatible:
        raise RuntimeError('Checkpoint không có tensor tương thích với architecture DySample.')
    report = rebuilt.model.load_state_dict(compatible, strict=False)
    print(f'Rebuilt checkpoint load: {len(compatible):,} compatible tensors')
    print(f'Rebuilt missing keys: {len(report.missing_keys):,}')
    assert count_dysample(rebuilt.model) == DYSAMPLE_EXPECTED_COUNT == 2
    return rebuilt, 'rebuilt'

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy best.pt: {BEST_MODEL_PATH}')
best_model, checkpoint_load_mode = load_custom_dysample_checkpoint(BEST_MODEL_PATH)
dysample_count_best = count_dysample(best_model.model)
print('DySample IN BEST MODEL:', dysample_count_best)
assert dysample_count_best == DYSAMPLE_EXPECTED_COUNT == 2
print(f'Checkpoint load mode: {checkpoint_load_mode}')

## 8. Validation metrics và error analysis

Chỉ validation. Báo cáo metric overall, từng class và riêng `rider_without_helmet`; confusion matrix được đọc từ object của Ultralytics, không tự bịa số.

In [ ]:
def safe_f1(precision_value, recall_value):
    denominator = precision_value + recall_value
    return 2.0 * precision_value * recall_value / denominator if denominator else 0.0

def extract_metrics(split_metrics, split_name):
    summary = {
        'Precision': float(split_metrics.box.mp),
        'Recall': float(split_metrics.box.mr),
        'mAP50': float(split_metrics.box.map50),
        'mAP50-95': float(split_metrics.box.map),
    }
    summary['F1'] = safe_f1(summary['Precision'], summary['Recall'])
    summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision_value, recall_value, ap50_value, ap5095_value = split_metrics.box.class_result(class_id)
        class_summary = {
            'Precision': float(precision_value),
            'Recall': float(recall_value),
            'AP50': float(ap50_value),
            'AP50-95': float(ap5095_value),
        }
        class_summary['F1'] = safe_f1(class_summary['Precision'], class_summary['Recall'])
        summary['by_class'][class_name] = class_summary
    print(f'--- {split_name} overall ---')
    for metric_name in ('Precision', 'Recall', 'F1', 'mAP50', 'mAP50-95'):
        print(f'{metric_name:10s}: {summary[metric_name]:.4f}')
    print('--- Per-class metrics ---')
    for class_name, values in summary['by_class'].items():
        values_text = ' | '.join(f'{key}={values[key]:.4f}' for key in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95'))
        print(f'{class_name:24s} | {values_text}')
    return summary

metrics = best_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=True)
validation_summary = extract_metrics(metrics, 'E4 validation')
no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']
print('--- rider_without_helmet ---')
for metric_name in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95'):
    print(f'{metric_name:10s}: {no_helmet_metrics[metric_name]:.4f}')

test_metrics = None
test_summary = None
assert test_metrics is None and test_summary is None

In [ ]:
PLOT_SPECS = [
    ('results.png', 'Training results'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box PR curve'),
    ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'),
    ('BoxR_curve.png', 'Box Recall curve'),
]
plot_roots = [Path(getattr(metrics, 'save_dir', EXPERIMENT_DIR)), EXPERIMENT_DIR]
def find_plot(filename):
    candidates = []
    for root in plot_roots:
        if root.exists():
            candidates.extend(root.rglob(filename))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

for filename, title in PLOT_SPECS:
    plot_path = find_plot(filename)
    if plot_path is None:
        print(f'Không tìm thấy {filename}; không tự tạo plot thay thế.')
        continue
    plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
    plt.imshow(Image.open(plot_path))
    plt.axis('off')
    plt.title(f'{title} - {plot_path.parent.name}')
    plt.show()

def get_confusion_matrix_array(split_metrics):
    confusion_object = getattr(split_metrics, 'confusion_matrix', None)
    matrix = getattr(confusion_object, 'matrix', None)
    if matrix is None:
        return None
    matrix = np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix.shape == expected_shape else None

confusion_matrix_array = get_confusion_matrix_array(metrics)
if confusion_matrix_array is None:
    print('Không đọc được confusion matrix object trực tiếp; không tạo số liệu lỗi nhân tạo.')
else:
    # Ultralytics detection confusion matrix: rows=predicted, columns=actual.
    confusion_labels = CLASS_NAMES + ['background']
    confusion_table = pd.DataFrame(
        confusion_matrix_array,
        index=[f'predicted:{label}' for label in confusion_labels],
        columns=[f'actual:{label}' for label in confusion_labels],
    )
    display(confusion_table)
    label_to_index = {label: index for index, label in enumerate(confusion_labels)}

    def error_count(actual_label, predicted_label):
        return float(confusion_matrix_array[label_to_index[predicted_label], label_to_index[actual_label]])

    error_analysis = pd.DataFrame([
        {'Error': 'rider_without_helmet -> rider_with_helmet', 'Count': error_count('rider_without_helmet', 'rider_with_helmet')},
        {'Error': 'rider_without_helmet -> background', 'Count': error_count('rider_without_helmet', 'background')},
        {'Error': 'background -> rider_without_helmet', 'Count': error_count('background', 'rider_without_helmet')},
    ])
    display(error_analysis)

## 9. Complexity: parameters, GFLOPs, model size, latency và FPS

Đo trên cùng `imgsz=640`, GPU và tối đa 50 ảnh validation. Không giả định DySample nhẹ hơn; bảng dưới đây là số đo thực tế.

In [ ]:
valid_img_dir = Path('/content/DetectedHelmets/DataSet2/DataSet2/valid/images')
valid_images = sorted([str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG') for path in valid_img_dir.glob(pattern)])
if not valid_images:
    raise FileNotFoundError(f'Không tìm thấy validation images: {valid_img_dir}')

def extract_gflops(info):
    if isinstance(info, (tuple, list)):
        for value in reversed(info):
            try:
                return float(value)
            except (TypeError, ValueError):
                continue
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    return None

def model_gflops(yolo_model):
    return extract_gflops(yolo_model.info(verbose=True, imgsz=640))

def benchmark_inference(yolo_model, image_paths):
    for image_path in image_paths[:2]:
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
    elapsed = []
    for image_path in image_paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - start_time)
    milliseconds = float(np.mean(elapsed) * 1000.0)
    return milliseconds, 1000.0 / milliseconds if milliseconds else 0.0

speed_images = valid_images[:min(50, len(valid_images))]
if not BASELINE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy baseline best.pt: {BASELINE_BEST_MODEL_PATH}')
baseline_model_for_complexity = YOLO(str(BASELINE_BEST_MODEL_PATH))
baseline_ms, baseline_fps = benchmark_inference(baseline_model_for_complexity, speed_images)
dysample_ms, dysample_fps = benchmark_inference(best_model, speed_images)

complexity = pd.DataFrame([
    {
        'Model': 'YOLO26s',
        'Parameters': sum(parameter.numel() for parameter in baseline_model_for_complexity.model.parameters()),
        'GFLOPs': model_gflops(baseline_model_for_complexity),
        'Model size (MB)': BASELINE_BEST_MODEL_PATH.stat().st_size / (1024 ** 2),
        'Inference (ms/image)': baseline_ms,
        'FPS': baseline_fps,
    },
    {
        'Model': 'YOLO26s+DySample',
        'Parameters': sum(parameter.numel() for parameter in best_model.model.parameters()),
        'GFLOPs': model_gflops(best_model),
        'Model size (MB)': BEST_MODEL_PATH.stat().st_size / (1024 ** 2),
        'Inference (ms/image)': dysample_ms,
        'FPS': dysample_fps,
    },
])
display(complexity)

## 10. So sánh trực tiếp với baseline trên cùng validation set

In [ ]:
baseline_comparison_metrics = baseline_model_for_complexity.val(
    data=str(DATA_YAML_PATH),
    split=VALIDATION_SPLIT,
    imgsz=640,
    device=0,
    plots=False,
)
baseline_summary = extract_metrics(baseline_comparison_metrics, 'YOLO26s baseline')
dysample_summary = validation_summary

def comparison_values(summary):
    no_helmet = summary['by_class']['rider_without_helmet']
    return {
        'Precision': summary['Precision'],
        'Recall': summary['Recall'],
        'F1': summary['F1'],
        'mAP50': summary['mAP50'],
        'mAP50-95': summary['mAP50-95'],
        'NoHelmet Precision': no_helmet['Precision'],
        'NoHelmet Recall': no_helmet['Recall'],
        'NoHelmet F1': no_helmet['F1'],
        'NoHelmet AP50': no_helmet['AP50'],
        'NoHelmet AP50-95': no_helmet['AP50-95'],
    }

baseline_values = comparison_values(baseline_summary)
dysample_values = comparison_values(dysample_summary)
comparison = pd.DataFrame({
    'Metric': list(baseline_values.keys()),
    'YOLO26s': list(baseline_values.values()),
    'YOLO26s+DySample': [dysample_values[key] for key in baseline_values],
})
comparison['Delta'] = comparison['YOLO26s+DySample'] - comparison['YOLO26s']
display(comparison)

## 11. Quyết định E4 theo kết quả đo thực tế

Không hard-code kết quả. Ưu tiên giữ DySample nếu mAP50-95, NoHelmet AP50 hoặc NoHelmet Recall có lợi mà không làm cả ba tiêu chí cùng giảm; xem complexity để cân nhắc trade-off inference.

In [ ]:
priority_metrics = ['mAP50-95', 'NoHelmet AP50', 'NoHelmet Recall']
priority_improvements = {metric: dysample_values[metric] > baseline_values[metric] for metric in priority_metrics}
any_priority_improvement = any(priority_improvements.values())
joint_priority_drop = all(dysample_values[metric] < baseline_values[metric] for metric in priority_metrics)
e4_decision = 'KEEP DySample' if any_priority_improvement and not joint_priority_drop else 'REJECT DySample'
print(f'Priority improvements: {priority_improvements}')
print(f'E4 decision: {e4_decision}')

## 12. Final audit trước khi lưu notebook

In [ ]:
assert MODEL_NAME == 'yolo26s.pt'
assert ultralytics.__version__ == ULTRALYTICS_VERSION == '8.4.155'
assert DATA_YAML_PATH == Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG == {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15, 'pretrained': True,
}
assert VALIDATION_SPLIT == 'val'
assert DYSAMPLE_EXPECTED_COUNT == 2
assert len(upsample_specs) == 2
assert {spec['level'] for spec in upsample_specs} == {'P5 -> P4', 'P4 -> P3'}
assert all(spec['input_stride'] == spec['output_stride'] * 2 for spec in upsample_specs)
assert all(spec['style'] == 'lp' and spec['dyscope'] is True for spec in upsample_specs)
assert count_dysample(custom_pytorch_model) == 2
assert trainer_dysample_count == 2
assert dysample_count_best == 2
assert trainer.model is custom_pytorch_model
assert test_metrics is None and test_summary is None
assert EXPERIMENT_NAME == 'helmet_yolo26s_dysample_dataset2'
assert BASELINE_EXPERIMENT_NAME not in str(BEST_MODEL_PATH)
assert BEST_MODEL_PATH == OUTPUT_DIR / 'helmet_yolo26s_dysample_dataset2' / 'weights' / 'best.pt'
assert EXPERIMENT_VERSION_FILE.exists()
assert ULTRALYTICS_VERSION in EXPERIMENT_VERSION_FILE.read_text(encoding='utf-8')
assert sum(1 for module in best_model.model.model if module.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}) == DETECT_HEAD_COUNT

forbidden_modules = {'P2', 'iAFF', 'MultiSEAM', 'SEAM', 'SCConv', 'CBAM', 'ECA', 'AFPN', 'ADown'}
custom_module_names = {submodule.__class__.__name__ for submodule in best_model.model.modules()}
assert not (custom_module_names & forbidden_modules), custom_module_names & forbidden_modules

print('Final audit passed.')
print(f'Ultralytics version: {ultralytics.__version__}')
print(f'DySample modules before/train/best: {dysample_before_train}/{trainer_dysample_count}/{dysample_count_best}')
print(f'Runtime upsample points: {[(spec["level"], spec["index"], spec["input_channels"], spec["groups"]) for spec in upsample_specs]}')
print(f'Validation only: split={VALIDATION_SPLIT!r}; test disabled')
print(f'E4 checkpoint: {BEST_MODEL_PATH}')
print(f'Baseline untouched: {BASELINE_BEST_MODEL_PATH}')
print(f'Decision from measured metrics: {e4_decision}')